# PyMIF notebook 07 - standalone label-only Zarr at the root

This notebook creates a **new OME-Zarr dataset whose root is the label image itself**.

This is intentionally different from notebook 06.

Notebook 06 shows labels attached to an existing image:

```text
image.zarr/
  0, 1, 2, ...
  labels/nuclei/0, 1, 2, ...
```

This notebook creates a standalone label dataset:

```text
nuclei_labels.zarr/
  0, 1, 2, ...
```

There is **no intensity image** in the store and there is **no `/labels/<name>` group**. The root multiscale dataset has `data_type="label"`.


In [1]:
from pathlib import Path
import shutil
import sys

import numpy as np
import zarr

# Use the installed package when available. When running from a local PyMIF
# source checkout, this fallback adds the repository root to sys.path.
try:
    import pymif.microscope_manager as mm
except ModuleNotFoundError:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "pymif").is_dir():
            sys.path.insert(0, str(candidate))
            break
    import pymif.microscope_manager as mm

OUTPUT_DIR = Path("pymif_tutorial_output")
OUTPUT_DIR.mkdir(exist_ok=True)
print("PyMIF managers loaded from:", mm.__file__)
print("Tutorial output folder:", OUTPUT_DIR.resolve())


PyMIF managers loaded from: C:\Users\nicol\Documents\GitHub\pymif\pymif\microscope_manager\__init__.py
Tutorial output folder: C:\Users\nicol\Documents\GitHub\pymif\examples\pymif_tutorial_output


In [2]:
def clean_path(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    return path


def summarize_manager(manager, title="dataset"):
    """Print the fields beginners usually need to inspect first."""
    print(f"\n{title}")
    print("-" * len(title))
    print("manager:", type(manager).__name__)
    print("axes:", manager.metadata.get("axes"))
    print("data_type:", manager.metadata.get("data_type"))
    print("is_label:", manager.metadata.get("is_label"))
    print("levels:", len(manager.data))
    for i, arr in enumerate(manager.data):
        print(
            f"  level {i}: shape={arr.shape}, "
            f"chunks={getattr(arr, 'chunksize', None)}, dtype={arr.dtype}"
        )
    for key in ["scales", "units", "time_increment", "time_increment_unit", "channel_names", "channel_colors"]:
        print(f"{key}:", manager.metadata.get(key))


def root_ome_attrs(path):
    """Return the OME/NGFF metadata dictionary from a Zarr root."""
    root = zarr.open_group(path, mode="r")
    attrs = root.attrs.asdict()
    return attrs.get("ome", attrs)  # v0.5 uses attrs["ome"]; v0.4 used attrs directly


def inspect_root_label_zarr(zarr_manager, title="root label Zarr"):
    """Check the important properties of a root-level label-only Zarr."""
    print(f"\n{title}")
    print("-" * len(title))
    print("root axes:", zarr_manager.metadata.get("axes"))
    print("root data_type:", zarr_manager.metadata.get("data_type"))
    print("root is_label:", zarr_manager.metadata.get("is_label"))
    print("root levels:", len(zarr_manager.data))
    print("root level 0 shape:", zarr_manager.data[0].shape)
    print("root level 0 dtype:", zarr_manager.data[0].dtype)
    print("root array keys:", sorted(zarr_manager.root.array_keys()))
    print("root group keys:", sorted(zarr_manager.root.group_keys()))
    print("label groups under /labels:", list(zarr_manager.labels.keys()))
    print("unique label IDs at level 0:", np.unique(zarr_manager.data[0].compute()))

    assert zarr_manager.metadata.get("data_type") == "label"
    assert "labels" not in zarr_manager.root
    assert len(zarr_manager.labels) == 0


## Step 1 - Create a toy segmentation label image

Most segmentation results should use integer labels and omit the channel axis.

Here the label array has axes `tzyx`:

- `t`: time
- `z`: plane
- `y`, `x`: image coordinates

Pixel value `0` means background. Positive integers are object IDs.


In [33]:
labels = np.zeros((1, 8, 128, 128), dtype=np.uint32)  # T, Z, Y, X

labels[:, 1:6, 18:45, 20:55] = 1
labels[:, 0:3, 25:70, 90:118] = 3

print("label array shape:", labels.shape)
print("label array dtype:", labels.dtype)
print("unique label IDs:", np.unique(labels))


label array shape: (1, 8, 128, 128)
label array dtype: uint32
unique label IDs: [0 1 3]


## Step 2 - Define metadata for a label-only root dataset

The important differences from an intensity image are:

- `data_type` is `"label"`.
- `is_label` is `True`.
- The dtype is an integer type such as `uint16`, `uint32`, or `uint64`.
- There is no channel axis, so `channel_names` and `channel_colors` are empty.

`scales` and `units` describe only the spatial axes, in the order they appear in `axes`. For `tzyx`, the spatial order is `z, y, x`.


In [34]:
label_metadata = {
    "name": "nuclei_labels_root", # optional, will be set to the Zarr group name if not provided
    "axes": "tzyx", # required, will be validated by ArrayManager and used to infer the dimension order of the array
    # "size": [labels.shape], # optional, will be inferred from the array shape if not provided
    "chunksize": [(1, 4, 64, 64)], # t, z, y, x chunk size [optional]
    "scales": [(1.0, 0.25, 0.25)],  # z, y, x spacing [optional]
    "units": ("micrometer", "micrometer", "micrometer"), # optional
    "time_increment": 1.0, # optional
    "time_increment_unit": "second", # optional
    "channel_names": [], # optional, can be set to a list of strings
    "channel_colors": [], # optional, can be set to a list of color strings (e.g. hex codes) or RGB tuples
    "dtype": "uint8",  # optional, will be inferred from the array dtype if not provided
    "data_type": "label",
    # "is_label": True, ### optional, will be set to True by ArrayManager when data_type is "label"
}

label_manager = mm.ArrayManager(
    labels,
    label_metadata,
    chunks=label_metadata["chunksize"][0],
)

summarize_manager(label_manager, "single-resolution label manager")



single-resolution label manager
-------------------------------
manager: ArrayManager
axes: tzyx
data_type: label
is_label: None
levels: 1
  level 0: shape=(1, 8, 128, 128), chunks=(1, 4, 64, 64), dtype=uint32
scales: [(1.0, 0.25, 0.25)]
units: ('micrometer', 'micrometer', 'micrometer')
time_increment: 1.0
time_increment_unit: second
channel_names: []
channel_colors: []


## Step 3 - Build a label pyramid

Label images should be downsampled with nearest-neighbor behavior so label IDs remain integers.

`downscale_factor=(1, 2, 2)` means keep `z` unchanged and downsample `y` and `x` by 2 at each pyramid level.


In [36]:
label_manager.build_pyramid(
    num_levels=3,
    downscale_factor=(1, 2, 2),  # z, y, x
)

summarize_manager(label_manager, "multiscale label manager")
for level, arr in enumerate(label_manager.data):
    print(f"level {level} unique IDs:", np.unique(arr.compute()))



multiscale label manager
------------------------
manager: ArrayManager
axes: tzyx
data_type: label
is_label: None
levels: 3
  level 0: shape=(1, 8, 128, 128), chunks=(1, 4, 64, 64), dtype=uint32
  level 1: shape=(1, 8, 64, 64), chunks=(1, 4, 32, 32), dtype=uint32
  level 2: shape=(1, 8, 32, 32), chunks=(1, 4, 16, 16), dtype=uint32
scales: [(1.0, 0.25, 0.25), (1.0, 0.5, 0.5), (1.0, 1.0, 1.0)]
units: ('micrometer', 'micrometer', 'micrometer')
time_increment: 1.0
time_increment_unit: second
channel_names: []
channel_colors: []
level 0 unique IDs: [0 1 3]
level 1 unique IDs: [0 1 3]
level 2 unique IDs: [0 1 3]


## Option A - Write the label image directly to the Zarr root

This is the simplest pattern when you already have the complete label array in memory or as a Dask array.

Because `label_manager.metadata["data_type"] == "label"`, `to_zarr(...)` writes a root-level label multiscale dataset. It does **not** create `/labels/nuclei`.


In [37]:
root_label_out = clean_path(OUTPUT_DIR / "standalone_label_root.zarr")

label_manager.to_zarr(
    root_label_out,
    ngff_version="0.5",
    # zarr_format=3,
)

print("Wrote standalone root label Zarr:", root_label_out)


Wrote standalone root label Zarr: pymif_tutorial_output\standalone_label_root.zarr


## Step 4 - Inspect the root store layout

The root should contain only pyramid arrays such as `0`, `1`, and `2`. It should not contain a `/labels` group.


In [38]:
root = zarr.open_group(root_label_out, mode="r")
ome = root_ome_attrs(root_label_out)
multiscales = ome["multiscales"][0]

print("root array keys:", sorted(root.array_keys()))
print("root group keys:", sorted(root.group_keys()))
print("root data_type:", ome.get("data_type"))
print("root multiscales type:", multiscales.get("type"))
print("dataset paths:", [dataset["path"] for dataset in multiscales["datasets"]])
print("root image-label metadata:", ome.get("image-label"))

assert sorted(root.array_keys()) == ["0", "1", "2"]
assert sorted(root.group_keys()) == []
assert "labels" not in root
assert ome.get("data_type") == "label"
assert multiscales.get("type") == "label"


root array keys: ['0', '1', '2']
root group keys: []
root data_type: label
root multiscales type: label
dataset paths: ['0', '1', '2']
root image-label metadata: {'source': {'image': '../'}}


## Step 5 - Read the standalone label Zarr back

`ZarrManager` can read this store because the root is still a valid multiscale OME-Zarr dataset.

For historical API compatibility, `ZarrManager` still exposes the root dataset as `manager.raw`. In this notebook, that root dataset is **not** a raw intensity image; its metadata says `data_type="label"`.


In [40]:
root_label_zarr = mm.ZarrManager(root_label_out, mode="r")
inspect_root_label_zarr(root_label_zarr, "standalone label root read-back")


/
├── 0 (1, 8, 128, 128) uint32
├── 1 (1, 8, 64, 64) uint32
└── 2 (1, 8, 32, 32) uint32

SIZE: [(1, 8, 128, 128), (1, 8, 64, 64), (1, 8, 32, 32)]
CHUNKSIZE: [(1, 4, 64, 64), (1, 4, 32, 32), (1, 4, 16, 16)]
SCALES: [(1.0, 0.25, 0.25), (1.0, 0.5, 0.5), (1.0, 1.0, 1.0)]
UNITS: ('micrometer', 'micrometer', 'micrometer')
TIME_INCREMENT: 1.0
TIME_INCREMENT_UNIT: second
CHANNEL_NAMES: []
CHANNEL_COLORS: []
DTYPE: uint32
PLANE_FILES: None
AXES: tzyx
DATA_TYPE: label
IS_LABEL: True
NGFF_VERSION: 0.5
ZARR_FORMAT: 3

standalone label root read-back
-------------------------------
root axes: tzyx
root data_type: label
root is_label: True
root levels: 3
root level 0 shape: (1, 8, 128, 128)
root level 0 dtype: uint32
root array keys: ['0', '1', '2']
root group keys: []
label groups under /labels: []
unique label IDs at level 0: [0 1 3]


## Option B - Create an empty label-only root Zarr, then write labels into `/`

Use this when you want to create the Zarr layout first and fill it later.

The key detail is `group="/"` in `write_label_region`. That tells PyMIF to write into the root label dataset, not into `/labels/nuclei`.


In [ ]:
empty_root_label_out = clean_path(OUTPUT_DIR / "standalone_label_root_empty_then_write.zarr")

# Reuse the multiscale metadata created by label_manager.build_pyramid().
# It contains size, chunksize, and scale entries for every pyramid level.
empty_label_zarr = mm.ZarrManager(
    empty_root_label_out,
    mode="w",
    metadata=label_manager.metadata,
    ngff_version="0.5",
    zarr_format=3,
)

# Fill the root label pyramid from the level-0 label image.
empty_label_zarr.write_label_region(
    labels,
    # group="/",
    # level=0,
    # downscale_factor=(1, 2, 2),
)

# Re-read so the manager reflects what is on disk.
empty_label_zarr = mm.ZarrManager(empty_root_label_out, mode="r")
inspect_root_label_zarr(empty_label_zarr, "empty root label Zarr after write")


/
├── 0 (1, 8, 128, 128) uint8
├── 1 (1, 8, 64, 64) uint8
└── 2 (1, 8, 32, 32) uint8

SIZE: [(1, 8, 128, 128), (1, 8, 64, 64), (1, 8, 32, 32)]
CHUNKSIZE: [(1, 4, 64, 64), (1, 4, 32, 32), (1, 4, 16, 16)]
SCALES: [(1.0, 0.25, 0.25), (1.0, 0.5, 0.5), (1.0, 1.0, 1.0)]
UNITS: ('micrometer', 'micrometer', 'micrometer')
TIME_INCREMENT: 1.0
TIME_INCREMENT_UNIT: second
CHANNEL_NAMES: []
CHANNEL_COLORS: []
DTYPE: uint8
PLANE_FILES: None
AXES: tzyx
DATA_TYPE: label
IS_LABEL: True
NGFF_VERSION: 0.5
ZARR_FORMAT: 3
/
├── 0 (1, 8, 128, 128) uint8
├── 1 (1, 8, 64, 64) uint8
└── 2 (1, 8, 32, 32) uint8

SIZE: [(1, 8, 128, 128), (1, 8, 64, 64), (1, 8, 32, 32)]
CHUNKSIZE: [(1, 4, 64, 64), (1, 4, 32, 32), (1, 4, 16, 16)]
SCALES: [(1.0, 0.25, 0.25), (1.0, 0.5, 0.5), (1.0, 1.0, 1.0)]
UNITS: ('micrometer', 'micrometer', 'micrometer')
TIME_INCREMENT: 1.0
TIME_INCREMENT_UNIT: second
CHANNEL_NAMES: []
CHANNEL_COLORS: []
DTYPE: uint8
PLANE_FILES: None
AXES: tzyx
DATA_TYPE: label
IS_LABEL: True
NGFF_VERSION: 0.5
Z

## Common mistakes

1. Accidentally creating `/labels/nuclei`: that is the notebook 06 pattern. Use it when a label belongs to an existing intensity image.
2. Keeping a channel axis for labels: most segmentation label images should use `tzyx`, `zyx`, or `yx`, not `tczyx`.
3. Using a float dtype for labels: label datasets must use integer dtypes.
4. Forgetting that `0` is background: keep object IDs positive and reserve `0` for background.


## Beginner takeaway

Use this root-only pattern for standalone label datasets:

```python
label_manager = mm.ArrayManager(labels, label_metadata, chunks=(1, 4, 48, 64))
label_manager.build_pyramid(num_levels=3, downscale_factor=(1, 2, 2))
label_manager.to_zarr("nuclei_labels.zarr", ngff_version="0.5", zarr_format=3)
```

Use notebook 06 instead when you already have an intensity image Zarr and want to attach one or more label images under `/labels/<label_name>`.
